In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded! ✅")

Libraries loaded! ✅


In [3]:
import os

# This shows you exactly what files are available
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/home-credit-default-risk/sample_submission.csv
/kaggle/input/competitions/home-credit-default-risk/bureau_balance.csv
/kaggle/input/competitions/home-credit-default-risk/POS_CASH_balance.csv
/kaggle/input/competitions/home-credit-default-risk/application_train.csv
/kaggle/input/competitions/home-credit-default-risk/HomeCredit_columns_description.csv
/kaggle/input/competitions/home-credit-default-risk/application_test.csv
/kaggle/input/competitions/home-credit-default-risk/previous_application.csv
/kaggle/input/competitions/home-credit-default-risk/credit_card_balance.csv
/kaggle/input/competitions/home-credit-default-risk/installments_payments.csv
/kaggle/input/competitions/home-credit-default-risk/bureau.csv


In [4]:
df = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/application_train.csv')
print("Shape:", df.shape)

Shape: (307511, 122)


In [5]:
target = df['TARGET']
df = df.drop('TARGET', axis=1)

print("Target shape:", target.shape)
print("Features shape:", df.shape)

Target shape: (307511,)
Features shape: (307511, 121)


In [6]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()

print(f"Numerical columns: {len(num_cols)}")
print(f"Categorical columns: {len(cat_cols)}")

Numerical columns: 105
Categorical columns: 16


In [7]:
# Fill numerical with median
num_imputer = SimpleImputer(strategy='median')
df[num_cols] = num_imputer.fit_transform(df[num_cols])

# Fill categorical with most frequent
cat_imputer = SimpleImputer(strategy='most_frequent')
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

print("Missing values after imputation:")
print(df.isnull().sum().sum())  # Should print 0

Missing values after imputation:
0


In [8]:
le = LabelEncoder()

for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

print("Categorical encoding done! ✅")
print("Shape after encoding:", df.shape)

Categorical encoding done! ✅
Shape after encoding: (307511, 121)


In [9]:
# Age in years (DAYS_BIRTH is negative)
df['AGE_YEARS'] = (-df['DAYS_BIRTH'] / 365).astype(int)

# Employment years
df['EMPLOYMENT_YEARS'] = (-df['DAYS_EMPLOYED'].clip(upper=0) / 365)

# Credit to Income ratio
df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / (df['AMT_INCOME_TOTAL'] + 1)

# Annuity to Income ratio
df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / (df['AMT_INCOME_TOTAL'] + 1)

print("New features created! ✅")
print("Shape after feature engineering:", df.shape)

New features created! ✅
Shape after feature engineering: (307511, 125)


In [11]:
scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

print("Scaling done! ✅")

Scaling done! ✅


In [12]:
df['TARGET'] = target

print("Final dataset shape:", df.shape)
print("Target distribution:")
print(df['TARGET'].value_counts())

# Save preprocessed data
df.to_csv('preprocessed_data.csv', index=False)
print("\nPreprocessed data saved! ✅")

Final dataset shape: (307511, 126)
Target distribution:
TARGET
0    282686
1     24825
Name: count, dtype: int64

Preprocessed data saved! ✅


In [13]:
print("=== Preprocessing Summary ===")
print(f"Final Shape: {df.shape}")
print(f"Missing Values: {df.isnull().sum().sum()}")
print(f"Numerical Features: {len(num_cols)}")
print(f"Categorical Features: {len(cat_cols)}")
print(f"New Engineered Features: 4")
print(f"Target Distribution:\n{df['TARGET'].value_counts()}")
print("\nPreprocessing Complete! ✅")

=== Preprocessing Summary ===
Final Shape: (307511, 126)
Missing Values: 0
Numerical Features: 105
Categorical Features: 16
New Engineered Features: 4
Target Distribution:
TARGET
0    282686
1     24825
Name: count, dtype: int64

Preprocessing Complete! ✅
